# High level notes

In [2]:
import tensorflow as tf
tf.compat.v1.enable_eager_execution()

In [3]:
from migration.datasets import create_AIS_dataset
from migration.models import vrnn

In [4]:
batch_size = 32
latent_size = 64

In [5]:
import tensorflow as tf
inputs, targets, _, _, _, lengths, mean =  create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                   '../../data/ct_2017010203_10_20/mean.pkl',
                   batch_size,
                   99999, # not used lol
                   300,
                   300, 
                   30,
                   72, 
                   shuffle=False,
                   repeat=False)



Instructions for updating:
tf.py_func is deprecated in TF V2. Instead, there are two
    options available in V2.
    - tf.py_function takes a python function which manipulates tf eager
    tensors instead of numpy arrays. It's easy to convert a tf eager tensor to
    an ndarray (just call tensor.numpy()) but having access to eager tensors
    means `tf.py_function`s can use accelerators such as GPUs as well as
    being differentiable using a gradient tape.
    - tf.numpy_function maintains the semantics of the deprecated tf.py_func
    (it is not differentiable, and manipulates numpy arrays). It drops the
    stateful argument making all functions stateful.
    
Instructions for updating:
Use `tf.cast` instead.
Instructions for updating:
Use `tf.cast` instead.
Instructions for updating:
Use `for ... in dataset:` to iterate over a dataset. If using `tf.estimator`, return the `Dataset` object directly from your input function. As a last resort, you can use `tf.compat.v1.data.make_one_

2025-07-03 12:19:17.917117: I tensorflow/core/platform/cpu_feature_guard.cc:142] Your CPU supports instructions that this TensorFlow binary was not compiled to use: AVX2 FMA
2025-07-03 12:19:17.921893: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 2688010000 Hz
2025-07-03 12:19:17.922828: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x2364dd80 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2025-07-03 12:19:17.922855: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Host, Default Version


In [6]:
generative_bias_init = -tf.math.log(1. / tf.clip_by_value(mean, 0.0001, 0.9999) - 1)
generative_distribution_class = vrnn.ConditionalBernoulliDistribution
model = vrnn.create_vrnn(inputs.get_shape().as_list()[2],
                            latent_size,
                            generative_distribution_class,
                            generative_bias_init=generative_bias_init,
                            raw_sigma_bias=0.5)

Instructions for updating:
This class is equivalent as tf.keras.layers.LSTMCell, and will be replaced by that in Tensorflow 2.0.


In [7]:
cell = model
seq_lengths = lengths
num_samples = 1

parallel_iterations=30
swap_memory=True

In [8]:
import migration.nested_utils as nested

In [15]:
batch_size = tf.shape(input=seq_lengths)[0]
max_seq_len = tf.reduce_max(input_tensor=seq_lengths)

# shape (t, B) of 1 and 0
seq_mask = tf.transpose(
        a=tf.sequence_mask(seq_lengths, maxlen=max_seq_len, dtype=tf.float32),
        perm=[1, 0])
# not accessed
# if num_samples > 1:
#     inputs, seq_mask = nested.tile_tensors([inputs, seq_mask], [1, num_samples])
# not accessed

# tensorarray of len t, elem (B, A)
inputs_ta, mask_ta = nested.tas_for_tensors([inputs, seq_mask], max_seq_len)

t0 = tf.constant(0, tf.int32)
init_states = cell.zero_state(batch_size * num_samples, # = 32
                              tf.float32)

# reads both in t0 (0)
init_inputs, init_mask = nested.read_tas([inputs_ta, mask_ta], t0)
ta_names = ['log_weights', 'log_ess']
tas = [tf.TensorArray(tf.float32, max_seq_len, name='%s_ta' % n)
            for n in ta_names]
log_weights_acc = tf.zeros([num_samples, batch_size], dtype=tf.float32)
kl_acc = tf.zeros([num_samples * batch_size], dtype=tf.float32)
accs = (log_weights_acc, kl_acc)

### inspecting variables

In [19]:
mask_ta.read(0)

InvalidArgumentError: Could not read index 0 twice because it was cleared after a previous read (perhaps try setting clear_after_read = false?)

In [91]:
inputs_ta_el.shape

(32, 702)

### init_states

In [48]:
with tf.Session() as sess:
    init_states = sess.run(init_states)

In [61]:
len(init_states[0])

2

The 64 is the output dimension of feature extractor (MLP with size 64 (which is the latent size size))

In [64]:
print(init_states[0][0].shape)
init_states[0][0]

(32, 64)


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [56]:
init_states[1].shape

(32, 64)

In [ ]:
assert (init_states[0][0] == 0).all()
assert (init_states[0][1] == 0).all()

assert (init_states[1] == 0).all() # tf.zeros( (batch_size, latent_size) )

"shape" ( (32, 64)
          (
            (32, 64),
            (32, 64)
        )

## ACCS

In [74]:
with tf.Session() as sess:
    accs = sess.run(accs)

In [83]:
print(accs[0].shape)
print(accs[1].shape)
assert [(accs[i] == 0).all() for i in range(0,2)]

(1, 32)
(32,)


In [85]:
def while_predicate(t, *unused_args):
    return t < max_seq_len

def while_step(t, rnn_state, tas, accs):
    """Implements one timestep of IWAE computation."""
    log_weights_acc, kl_acc = accs
    cur_inputs, cur_mask = nested.read_tas([inputs_ta, mask_ta], t)
    # Run the cell for one step.
    log_q_z, log_p_z, log_p_x_given_z, kl, new_state, new_rnn_out\
                                                    = cell(cur_inputs,
                                                        rnn_state,
                                                        cur_mask,
                                                        )
    # Compute the incremental weight and use it to update the current
    # accumulated weight.
    kl_acc += kl * cur_mask
    log_alpha = (log_p_x_given_z + log_p_z - log_q_z) * cur_mask
    log_alpha = tf.reshape(log_alpha, [num_samples, batch_size])
    log_weights_acc += log_alpha 
    # Calculate the effective sample size.
    ess_num = 2 * tf.reduce_logsumexp(input_tensor=log_weights_acc, axis=0)
    ess_denom = tf.reduce_logsumexp(input_tensor=2 * log_weights_acc, axis=0)
    log_ess = ess_num - ess_denom
    # Update the  Tensorarrays and accumulators.
    ta_updates = [log_weights_acc, log_ess]
    new_tas = [ta.write(t, x) for ta, x in zip(tas, ta_updates)]
    new_accs = (log_weights_acc, kl_acc)
    return t + 1, new_state, new_tas, new_accs

## unwarping while loop

In [ ]:
t = t0
rnn_state = init_states
tas = tas
accs = accs

# unlaod the weight
log_weights_acc, kl_acc = accs
# unload the input and mask (a (32,64) and (32,) tensor)
cur_inputs, cur_mask = nested.read_tas([inputs_ta, mask_ta], t)


In [90]:
cur_mask

<tf.Tensor 'TensorArrayReadV3_14:0' shape=(?,) dtype=float32>

In [ ]:
@tf.function
def run():
    log_q_z, log_p_z, log_p_x_given_z, kl, new_state, new_rnn_out = cell(cur_inputs, rnn_state, cur_mask)
    return log_q_z, log_p_z, log_p_x_given_z, kl, new_state, new_rnn_out
# Run the cell for one step.

In [99]:
_, _, tas, accs = tf.while_loop(cond=while_predicate,
                                body=while_step,
                                loop_vars=(t0, init_states, tas, accs),
                                parallel_iterations=parallel_iterations,
                                swap_memory=swap_memory)


OperatorNotAllowedInGraphError: iterating over `tf.Tensor` is not allowed in Graph execution. Use Eager execution or decorate this function with @tf.function.

originally defined at:
  File "/tmp/ipykernel_4819/1835667962.py", line 7, in <module>
    raw_sigma_bias=0.5)
  File "/workspaces/GeoTrackNet/migration/packages/v1/src/migration/models/vrnn.py", line 347, in create_vrnn
    prior, approx_posterior, generative, random_seed=random_seed)
  File "/workspaces/GeoTrackNet/migration/packages/v1/src/migration/models/vrnn.py", line 123, in __init__
    super(VRNNCell, self).__init__(name=name)
  File "/workspaces/GeoTrackNet/migration/packages/v1/.venv/lib/python3.7/site-packages/sonnet/python/modules/base.py", line 182, in __init__
    custom_getter_=self._custom_getter)
  File "/workspaces/GeoTrackNet/migration/packages/v1/.venv/lib/python3.7/site-packages/tensorflow_core/python/ops/template.py", line 161, in make_template
    **kwargs)


In [ ]:

## Here log_weights is acc log_weights
log_weights, log_ess = [x.stack() for x in tas]
final_log_weights, kl = accs
log_p_hat = (tf.reduce_logsumexp(input_tensor=final_log_weights, axis=0) -
                                tf.math.log(tf.cast(num_samples, dtype=tf.float32)))
kl = tf.reduce_mean(input_tensor=tf.reshape(kl, [num_samples, batch_size]), axis=0)
log_weights = tf.transpose(a=log_weights, perm=[0, 2, 1])
return log_p_hat, kl, log_weights, log_ess